In [1]:
import os
import glob
from docx import Document
from tqdm import tqdm  # Use tqdm.notebook for Jupyter Notebooks
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

def read_docx(file_path):
    doc = Document(file_path)
    lines = []
    for para in doc.paragraphs:
        para_lines = para.text.split('\n')
        lines.extend(para_lines)
    return lines

def extract_subject_from_filename(filename):
    if 'Maths' in filename:
        return 'Maths Statement or Equation'
    elif 'Chemistry' in filename:
        return 'Chemistry Statement'
    elif 'General' in filename:
        return 'General Statement'

def generate_questions_batch(scenarios, generate_text, subject, batch_size=5):
    questions_batch = []
    total_scenarios = len(scenarios)
    for i in tqdm(range(0, total_scenarios, batch_size), desc="Generating Questions", leave=False):
        batch = scenarios[i:i+batch_size]
        prompts = [
            f"""[INST]Imagine you are a human, this is the first time you are coming across this {subject}, you have no previous knowledge of it "{scenario}", what are the top 5 questions that would pop up in your head which would be most useful in learning about it as you are new to it. Give me a simple bullet point list, don't explain them or expand them[/INST]"""
            for scenario in batch
        ]
        results = generate_text(prompts)
        for result in results:
            questions_batch.append(result[0]["generated_text"])

    return questions_batch

def write_questions_to_file(questions, output_file_path):
    with open(output_file_path, "a", encoding="utf-8") as file:
        for question in questions:
            file.write(question + "\n\n")

def process_directory(input_directory, output_directory, batch_size=5):
    os.makedirs(output_directory, exist_ok=True)
    
    tokenizer = AutoTokenizer.from_pretrained("13b/models--meta-llama--Llama-2-13b-chat-hf/snapshots/c2f3ec81aac798ae26dcc57799a994dfbf521496", local_files_only=True)
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained("13b/models--meta-llama--Llama-2-13b-chat-hf/snapshots/c2f3ec81aac798ae26dcc57799a994dfbf521496", local_files_only=True, quantization_config=bnb_config)
    generate_text = pipeline(model=model, tokenizer=tokenizer, return_full_text=True, task='text-generation', temperature=0.1, max_new_tokens=128, repetition_penalty=1.1)
    
    docx_files = glob.glob(os.path.join(input_directory, '*.docx'))
    total_files = len(docx_files)
    for file_path in tqdm(docx_files, desc="Processing Files"):
        subject = extract_subject_from_filename(os.path.basename(file_path))
        lines = read_docx(file_path)
        tqdm.write(f"Processing {os.path.basename(file_path)} with {len(lines)} sentences. Subject: {subject}")
        questions = generate_questions_batch(lines, generate_text, subject, batch_size=batch_size)
        
        output_file_name = os.path.basename(file_path).replace('.docx', '_results.txt')
        output_file_path = os.path.join(output_directory, output_file_name)
        
        write_questions_to_file(questions, output_file_path)
        tqdm.write(f"Finished processing {file_path}. Results appended to {output_file_path}\n")

# Example usage
process_directory('Question_data_2', 'delete_this', batch_size=5)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Processing Files:   0%|          | 0/7 [00:00<?, ?it/s]

Processing Chemistry - Intermediate.docx with 161 sentences. Subject: Chemistry Statement



Generating Questions:  30%|███       | 10/33 [05:04<11:45, 30.68s/it]/home/sjavaji_umass_edu/.local/lib/python3.9/site-packages/transformers/pipelines/base.py:1123: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(

Processing Files:  14%|█▍        | 1/7 [16:26<1:38:39, 986.66s/it]   

Finished processing Question_data_2/Chemistry - Intermediate.docx. Results appended to delete_this/Chemistry - Intermediate_results.txt

Processing Chemistry BASIC.docx with 161 sentences. Subject: Chemistry Statement


...

Finished processing Question_data_2/Maths - BASIC statements.docx. Results appended to delete_this/Maths - BASIC statements_results.txt

Processing Maths - Intermediate .docx with 108 sentences. Subject: Maths Statement or Equation



Processing Files:  86%|████████▌ | 6/7 [1:41:10<15:18, 918.38s/it]   

Finished processing Question_data_2/Maths - Intermediate .docx. Results appended to delete_this/Maths - Intermediate _results.txt

Processing Maths -Advance.docx with 101 sentences. Subject: Maths Statement or Equation



Processing Files: 100%|██████████| 7/7 [1:51:09<00:00, 952.75s/it]   

Finished processing Question_data_2/Maths -Advance.docx. Results appended to delete_this/Maths -Advance_results.txt

